This should be run before the labeling session to produce all of the necessary files

In [2]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import json

from torch.utils.data import DataLoader
import torch

from tqdm import tqdm
import pandas as pd
import nibabel as nib
from matplotlib.backends.backend_pdf import PdfPages

In [3]:
pdf_dir = Path('/data/vision/polina/users/marcusbl/bin_class/label_sessions_data/label_session_3-11')
pdf_dir.mkdir(exist_ok=True)

In [4]:
old_label_session_samples = pd.read_csv('/data/vision/polina/users/marcusbl/bin_class/' \
                                        'label_sessions_data/label_session_1-15/samples.csv', index_col = 0)

# Create Stack DataFrames

The goal of this section is to produce a big dataframe containg all of the samples across the two datasets, with all of their current and old labels. 

This includes any previous data labeling session, stored in the `label_sessions_data` folder


**DataFrame Features**
- Brain Type (None if unknown)
- Path to stack
- Path to mask
- Person 
- Dataset
- MAP ID
- Gestinational Age
- Old Label (None if n/t) 
- Person ID

## BCH Stack Dataset

In [5]:
bch_path = Path('/data/vision/polina/users/mfirenze/Data_sharing_MIT_Margherita')

In [6]:
bch_info_path = bch_path / 'marcus_info.csv'
bch_info2_path = bch_path / 'data_list.csv'

bch_df = pd.read_csv(bch_info_path)
bch_bonus_info = pd.read_csv(bch_info2_path)

Now we combine all of the BCH info into a big dataframe

In [7]:
# 1. Replace beginning path of data locations
bch_df['path'] = str(bch_path) + bch_df['Data Location'].str.split('mnt1').str[1]
bch_df = bch_df.drop(columns = 'Data Location')

# 2. Get the person
bch_df["person"] = bch_df["path"].str.extract(r"(?:processed|failed)/(.*?)/raw/")

# 3. Get the mask location 
bch_df["mask_path"] = (
    bch_df['path'].str.replace("/raw/", "/masks/", regex=False)
    .str.replace(r"\.nii$", "_mask.nii", regex=True)
)

# 4. Add a flag for dataset
bch_df['dataset'] = 'BCH'

# 5. Get the GA
bch_df['MAP ID'] = bch_df['path'].str.extract(r'(MAP-[^/]+)')
bch_df = bch_df.merge(
    bch_bonus_info[['MAP ID', ' GA']],
    on='MAP ID',
    how='left'           
)
bch_df.rename(columns={' GA': 'GA'}, inplace=True)

In [8]:
bch_df.head()

,Brain Type,path,person,mask_path,dataset,MAP ID,GA
0,axi,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-B510/MR-EI_Fetal_Neuro-26241177-20211007,/data/vision/polina/users/mfirenze/Data_sharin...,BCH,MAP-B510,32.4
1,sag,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-B510/MR-EI_Fetal_Neuro-26241177-20211007,/data/vision/polina/users/mfirenze/Data_sharin...,BCH,MAP-B510,32.4
2,cor,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-B510/MR-EI_Fetal_Neuro-26241177-20211007,/data/vision/polina/users/mfirenze/Data_sharin...,BCH,MAP-B510,32.4
3,axi,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-B510/MR-EI_Fetal_Neuro-26241177-20211007,/data/vision/polina/users/mfirenze/Data_sharin...,BCH,MAP-B510,32.4
4,sag,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-B510/MR-EI_Fetal_Neuro-26241177-20211007,/data/vision/polina/users/mfirenze/Data_sharin...,BCH,MAP-B510,32.4


## Ramya Stack Dataset

In [9]:
ramya_path = Path('/data/vision/polina/users/marcusbl/data')

rows = []

for person_dir in ramya_path.iterdir():
    if not person_dir.is_dir():
        continue

    for stack_dir in person_dir.iterdir(): 
        if not stack_dir.is_dir():
            continue

        rows.append({
            'path': stack_dir / 'clean' / 'dicoms.npy',
            'Brain Type': None,
            'person': person_dir.stem,
            'dataset': "R",
            'mask_path': stack_dir / 'clean' / 'masks.npy',
            'MAP ID': None,
            'GA': None,
        })

ramya_df = pd.DataFrame(rows)

In [10]:
ramya_df.head()

,path,Brain Type,person,dataset,mask_path,MAP ID,GA
0,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,None
1,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,None
2,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,None
3,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,None
4,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,None


## Merge Stack DataFrames

In [11]:
assert sorted(ramya_df.columns) == sorted(bch_df.columns)
stack_df = pd.concat([ramya_df, bch_df])
stack_df = stack_df.reset_index(drop=True)

/tmp/ipykernel_1252528/2697115292.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stack_df = pd.concat([ramya_df, bch_df])


Add a person_id column that maps person -> person_id (int)

In [12]:
person_map = {person: i for i,person in enumerate(stack_df['person'].unique())}
stack_df['person_id'] = stack_df['person'].map(person_map)

In [13]:
stack_df.head(5)

,path,Brain Type,person,dataset,mask_path,MAP ID,GA,person_id
0,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0
1,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,1
2,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,1
3,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,1
4,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,1


In [14]:
stack_df.tail(5)

,path,Brain Type,person,dataset,mask_path,MAP ID,GA,person_id
712,/data/vision/polina/users/mfirenze/Data_sharin...,cor,MAP-C5104/MR-EI_Fetal_Neuro-26489856-20220602,BCH,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-C5104,29.3,96
713,/data/vision/polina/users/mfirenze/Data_sharin...,sag,MAP-C5104/MR-EI_Fetal_Neuro-26489856-20220602,BCH,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-C5104,29.3,96
714,/data/vision/polina/users/mfirenze/Data_sharin...,sag,MAP-C5110/MR-EI_Fetal_Body-26764701-20230223,BCH,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-C5110,36.1,97
715,/data/vision/polina/users/mfirenze/Data_sharin...,cor,MAP-C5110/MR-EI_Fetal_Body-26764701-20230223,BCH,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-C5110,36.1,97
716,/data/vision/polina/users/mfirenze/Data_sharin...,cor,MAP-C5110/MR-EI_Fetal_Body-26764701-20230223,BCH,/data/vision/polina/users/mfirenze/Data_sharin...,MAP-C5110,36.1,97


# Stack DF -> Slice DF

We now want to go through each stack and turn it into a bunch of slices, possibly w/ labels 

In [15]:
print(f"There are {len(stack_df)} stacks across all datasets")

There are 717 stacks across all datasets


## Stacks -> Slices

In [16]:
def parse_stack(stack_path: Path | str, mask_path: Path | str) -> pd.DataFrame:
    """
    Given:
    1 - stack path
    2 - mask path
    
    Returns: DataFrame of all the slices within the stack, along w/ min & max slice number 
    """
    stack_path = Path(stack_path)
    mask_path = Path(mask_path)

    # Download nifti data and mask data
    if stack_path.suffix == '.npy':
        nifti_data = np.load(stack_path)
    else:
        nifti_img = nib.load(stack_path)
        nifti_data = nifti_img.get_fdata()

    if mask_path.suffix == '.npy':
        mask_data = np.load(mask_path).astype(bool)
    else:
        mask_img = nib.load(mask_path)
        mask_data = mask_img.get_fdata().astype(bool)

    # Calculate the # of scans [ignore unmasked beginning and end]
    start, end = 100, 0
    for slice_num in range(nifti_data.shape[-1]):
        mask = mask_data[:, :, slice_num] 

        if mask.sum() > 0:
            start = min(start, slice_num)
            end = max(end, slice_num)

    total_slices_cnt = int(nifti_data.shape[-1]) 
    kept_slices_cnt = end - start + 1
    kept_slices = list(range(start, end + 1)) # [start, end] inclusive 


    return pd.DataFrame({
        'slice_num': kept_slices,
        'stack_slices_cnt': [kept_slices_cnt] * len(kept_slices),
        'full_stack_slices_cnt': [total_slices_cnt] * len(kept_slices),
        'stack_slices': [kept_slices] * len(kept_slices)
    })

Parse each stack and append to the slice DataFrame! This entire process should take around 2 minutes

In [17]:
all_slices = []

for _, row in tqdm(list(stack_df.iterrows())):
    new_df = parse_stack(row["path"], row["mask_path"])
    
    # attach original row columns to returned df
    for col in stack_df.columns:
        new_df[col] = row[col]
    
    all_slices.append(new_df)

100%|██████████| 717/717 [01:35<00:00,  7.54it/s]


In [18]:
slice_df = pd.concat(all_slices, ignore_index=True)
slice_df[['slice_num', 'stack_slices_cnt', 'full_stack_slices_cnt']] = slice_df[['slice_num', 'stack_slices_cnt', 'full_stack_slices_cnt']].astype('int32')

In [19]:
slice_df.head(3)

,slice_num,stack_slices_cnt,full_stack_slices_cnt,stack_slices,path,Brain Type,person,dataset,mask_path,MAP ID,GA,person_id
0,3,34,42,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0
1,4,34,42,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0
2,5,34,42,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0


In [20]:
len(slice_df)

20212

## Add old labels to the dataset


In [21]:
labeled_slice_df = slice_df.copy()

Ramya's Dataset came w/ labels: **label_R**

In [22]:
# Ramya's Dataset Labels (label_R)

slice_df_R = labeled_slice_df[labeled_slice_df['dataset'] == 'R']

for idx, row in tqdm(slice_df_R.iterrows(), total=len(slice_df_R)):
    slice_path = Path(row['path'])
    slice_num = str(int(row['slice_num']))

    with open(slice_path.parent / 'labels.json') as f:
        labels_map = json.load(f)

    if slice_num in labels_map:
        labeled_slice_df.loc[idx, 'label_R'] = labels_map[slice_num]
    else:
        labeled_slice_df.loc[idx, 'label_R'] = None

100%|██████████| 8089/8089 [00:21<00:00, 381.68it/s]


In [23]:
labeled_slice_df.sort_values(by=['path', 'slice_num'])[['path', 'slice_num']].head(5)

,path,slice_num
7404,/data/vision/polina/users/marcusbl/data/anon-0...,1
7405,/data/vision/polina/users/marcusbl/data/anon-0...,2
7406,/data/vision/polina/users/marcusbl/data/anon-0...,3
7407,/data/vision/polina/users/marcusbl/data/anon-0...,4
7408,/data/vision/polina/users/marcusbl/data/anon-0...,5


Previous Data Labeling Sessesion: **label_i**

In [24]:
old_label_session_samples.sort_values(by=['path', 'scan_num'])[['path', 'scan_num']].head(5)

,path,scan_num
7311,/data/vision/polina/users/marcusbl/data/anon-0...,0
7312,/data/vision/polina/users/marcusbl/data/anon-0...,1
7313,/data/vision/polina/users/marcusbl/data/anon-0...,2
7314,/data/vision/polina/users/marcusbl/data/anon-0...,3
7315,/data/vision/polina/users/marcusbl/data/anon-0...,4


In [25]:
old_label_session_samples['path'] = old_label_session_samples['path'].astype(str)
labeled_slice_df['path'] = labeled_slice_df['path'].astype(str)

In [26]:
right_df = (
    old_label_session_samples
    .dropna(subset=['label'])
    .groupby(['path', 'scan_num'], as_index=False)['label']
    .agg(list)
    .rename(columns={'label': 'label_1'})
)

labeled_slice_df = (
    labeled_slice_df.merge(
        right_df,
        left_on=['path', 'slice_num'],
        right_on=['path', 'scan_num'],
        how='left'
    )
    .drop(columns='scan_num')
)

Let's see how many datapoints are labeled so far 

In [27]:
r_labeled = labeled_slice_df['label_R'].notna()
session1_labeled = labeled_slice_df['label_1'].dropna().apply(len)
at_least_one_label = (r_labeled > 0).astype(int) + (session1_labeled).astype(int)

print("R Labels: ", r_labeled.sum())
print("Total Data Session #1 labels:", session1_labeled.sum())
print("Total Datapoints w/ at least one label:", int(at_least_one_label.sum()))

R Labels:  7761
Total Data Session #1 labels: 11766
Total Datapoints w/ at least one label: 15203


# Generating the Mosaics for Next Session

At this point we have all of the datapoints and any labels that they have had in the past. Now we want to generate mosaics for the next round of data labeling

In [28]:
labeled_slice_df.head(3)

,slice_num,stack_slices_cnt,full_stack_slices_cnt,stack_slices,path,Brain Type,person,dataset,mask_path,MAP ID,GA,person_id,label_R,label_1
0,3,34,42,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0,0.0,"[1, 0]"
1,4,34,42,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0,0.0,"[1, 1]"
2,5,34,42,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0,0.0,"[1, 0]"


In [29]:
len(labeled_slice_df)

20212

## Choosing Which Stacks to Label

We want to choose stacks that haven't been labeled yet.

We have 98 subjects; 2 stacks per subject; 2 labelers per

Thus, a total of 396 stacks in the final list

In [30]:
final_stacks = []

In [31]:
has_label1 = labeled_slice_df['label_1'].notna()
result = has_label1.groupby(labeled_slice_df['path']).any()

unlabeled_stacks = result[result == False].index
len(unlabeled_stacks)

330

In [32]:
final_stacks = []

for person_id, person_group in labeled_slice_df.groupby('person_id'):
    person_stacks = person_group['path'].drop_duplicates()

    remaining_stacks = person_stacks[person_stacks.isin(unlabeled_stacks)]
    n = len(remaining_stacks)

    if n >= 2:
        chosen_stacks = remaining_stacks.sample(n=2, replace=False).tolist()
    else:
        other_stacks = person_stacks[~person_stacks.isin(remaining_stacks)]
        add_random_stacks = other_stacks.sample(
            n=min(2 - n, len(other_stacks)),
            replace=False
        ).tolist()

        chosen_stacks = remaining_stacks.tolist() + add_random_stacks

    final_stacks.extend(chosen_stacks)

final_stacks = np.array(final_stacks)
np.random.shuffle(final_stacks)

In [33]:
len(final_stacks)

194

## Generating Mosaics

In [34]:
stacks_per_pdf = 50
repeat_stack_cnt = 2

Use the `stack_df` and the `final_stacks` objects to create a `pd.DataFrame` of all stacks that are about to be labeled

In [35]:
display_df = stack_df[stack_df['path'].astype(str).isin(final_stacks)]
assert len(display_df) == len(final_stacks)
display_df.head(2)

,path,Brain Type,person,dataset,mask_path,MAP ID,GA,person_id
0,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00067-2-11-2015,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,0
3,/data/vision/polina/users/marcusbl/data/anon-0...,None,anon-00044-4-21-2017,R,/data/vision/polina/users/marcusbl/data/anon-0...,None,NaN,1


In [38]:
display_df.to_csv(pdf_dir / 'display_df.csv')
stack_df.to_csv(pdf_dir / 'stack_df.csv')
labeled_slice_df.to_csv(pdf_dir / 'labled_slice_df.csv')

Now go to your terminal and run `generate_mosaics.py`. Don't forget to adjust the `pdf_dir` and other arugments in the main block on the bottom